# Compute the rule-based baseline

**Purpose.** Assign each variant to one prioritized annotation stratum and apply the subgroup-prevalence baseline estimated on the complete manuscript benchmark.

## Reproducibility contract

- **Run from:** the repository root or this notebook's model directory.
- **Variant key:** `#CHROM`, `POS`, `REF`, and `ALT`.
- **Input:** `data/sample_data.csv.gz`.
- **Output:** `data/model_scores/rule_based.csv.gz`.
- **External data:** none.

The fixed subgroup priors below were estimated once from the complete 242,132-variant benchmark. Keeping them fixed makes the bundled sample reproduce the manuscript baseline exactly.


## 1. Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Support kernels started from either the repository root or this notebook's directory.
START_DIR = Path.cwd().resolve()
REPO_ROOT = START_DIR if (START_DIR / "data" / "sample_data.csv.gz").is_file() else START_DIR.parent
DATA_DIR = REPO_ROOT / "data"
MODEL_SCORE_DIR = DATA_DIR / "model_scores"
BENCHMARK_PATH = DATA_DIR / "sample_data.csv.gz"
OUTPUT_PATH = MODEL_SCORE_DIR / "rule_based.csv.gz"
VARIANT_KEY = ["#CHROM", "POS", "REF", "ALT"]

MODEL_SCORE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Compute scores


In [ ]:
SUBGROUP_COLS = [
    "group: coding",
    "group: missense",
    "group: missense + 3'UTR",
    "group: missense + intron (non-splice)",
    "group: stop gain",
    "group: start loss",
    "group: noncoding",
    "group: stop loss",
    "group: synonymous",
    "group: 5'UTR",
    "group: 3'UTR",
    "group: 3'UTR + RNA gene",
    "group: splice",
    "group: intron (non-splice)",
    "group: RNA gene",
]

PRIORITY_ORDER = [
    "group: missense + 3'UTR",
    "group: missense + intron (non-splice)",
    "group: 3'UTR + RNA gene",
    "group: stop gain",
    "group: start loss",
    "group: stop loss",
    "group: synonymous",
    "group: missense",
    "group: splice",
    "group: 5'UTR",
    "group: 3'UTR",
    "group: intron (non-splice)",
    "group: RNA gene",
    "group: noncoding",
    "group: coding",
]

# Pathogenic proportions estimated once on the complete manuscript benchmark.
STRATUM_SCORES = {
    "group: coding": 0.37300613496932516,
    "group: missense": 0.3576475575863206,
    "group: missense + 3'UTR": 0.47,
    "group: missense + intron (non-splice)": 0.2582781456953642,
    "group: stop gain": 0.9967882651838359,
    "group: start loss": 0.9759797724399494,
    "group: noncoding": 0.020305631149256854,
    "group: stop loss": 0.6451612903225806,
    "group: synonymous": 0.0052518621386188616,
    "group: 5'UTR": 0.013411567476948869,
    "group: 3'UTR": 0.0010967687505272926,
    "group: 3'UTR + RNA gene": 0.058823529411764705,
    "group: splice": 0.992923137409082,
    "group: intron (non-splice)": 0.01190873038680384,
    "group: RNA gene": 0.16,
}

if set(PRIORITY_ORDER) != set(SUBGROUP_COLS):
    raise ValueError("PRIORITY_ORDER and SUBGROUP_COLS must contain the same columns")

priority_indices = np.array(
    [SUBGROUP_COLS.index(column) for column in PRIORITY_ORDER], dtype=np.int64
)


def assign_stratum_index(group_matrix: np.ndarray):
    """Return the first matching subgroup according to PRIORITY_ORDER."""
    prioritized = group_matrix[:, priority_indices]
    has_group = prioritized.any(axis=1)
    first_position = np.argmax(prioritized, axis=1)
    assigned_index = np.full(len(group_matrix), -1, dtype=np.int32)
    assigned_index[has_group] = priority_indices[first_position[has_group]]
    return assigned_index, has_group


benchmark = pd.read_csv(BENCHMARK_PATH, dtype={"#CHROM": str}, low_memory=False)
required_columns = VARIANT_KEY + SUBGROUP_COLS
missing_columns = [column for column in required_columns if column not in benchmark.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

group_matrix = benchmark[SUBGROUP_COLS].fillna(0).to_numpy(dtype=np.int8)
assigned_index, has_group = assign_stratum_index(group_matrix)

stratum = pd.Series(pd.NA, index=benchmark.index, dtype="object")
stratum.loc[has_group] = np.asarray(SUBGROUP_COLS, dtype=object)[
    assigned_index[has_group]
]
benchmark["_rule_stratum"] = stratum
benchmark["Rule_based"] = benchmark["_rule_stratum"].map(STRATUM_SCORES)
benchmark.loc[~has_group, "Rule_based"] = np.nan

print(f"Variants: {len(benchmark):,}")
print(f"Variants without a rule-based score: {benchmark['Rule_based'].isna().sum():,}")
print(f"Scored strata: {benchmark['_rule_stratum'].nunique(dropna=True):,}")


## 3. Validate and save output


In [ ]:
output = benchmark[VARIANT_KEY + ["Rule_based"]].copy()

if len(output) != len(benchmark):
    raise RuntimeError("Output row count does not match the input row count")
if output[VARIANT_KEY].duplicated().any():
    raise ValueError("Duplicate variant keys were found in the output")

output.to_csv(OUTPUT_PATH, index=False, compression="gzip")
print(f"Saved {len(output):,} variants to {OUTPUT_PATH}")
